# PyMIF notebook 03 - create OME-Zarr with metadata

This notebook shows two ways to create an OME-Zarr:

1. Start from an existing manager and call `to_zarr`.
2. Create an empty on-disk Zarr from metadata first, then fill it with region writes.

The second approach is useful when you want to stream data tile-by-tile or append derived groups and labels later.

In [ ]:
from pathlib import Path
import shutil
import sys
import numpy as np

# Use the installed package when available. When running from a local PyMIF
# source checkout, this fallback adds the repository root to sys.path.
try:
    import pymif.microscope_manager as mm
except ModuleNotFoundError:
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / "pymif").is_dir():
            sys.path.insert(0, str(candidate))
            break
    import pymif.microscope_manager as mm

OUTPUT_DIR = Path("pymif_tutorial_output")
OUTPUT_DIR.mkdir(exist_ok=True)
print("PyMIF managers loaded from:", mm.__file__)
print("Tutorial output folder:", OUTPUT_DIR.resolve())

In [ ]:
def summarize_manager(manager, title="dataset"):
    """Print the fields beginners usually need to inspect first."""
    print(f"\n{title}")
    print("-" * len(title))
    print("manager:", type(manager).__name__)
    print("axes:", manager.metadata.get("axes"))
    print("data_type:", manager.metadata.get("data_type", "intensity"))
    print("levels:", len(manager.data))
    for i, arr in enumerate(manager.data):
        print(f"  level {i}: shape={arr.shape}, chunks={getattr(arr, 'chunksize', None)}, dtype={arr.dtype}")
    for key in ["scales", "units", "time_increment", "time_increment_unit", "channel_names", "channel_colors"]:
        print(f"{key}:", manager.metadata.get(key))


def clean_path(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    return path

In [ ]:
def make_small_raw_manager(seed=0, num_levels=2):
    """Create a tiny TCZYX intensity dataset for examples."""
    rng = np.random.default_rng(seed)
    raw = rng.integers(0, 1200, size=(2, 3, 4, 64, 64), dtype=np.uint16)
    metadata = {
        "name": "small_raw",
        "axes": "tczyx",
        "size": [raw.shape],
        "chunksize": [(1, 1, 2, 32, 32)],
        "scales": [(2.0, 0.30, 0.30)],   # z, y, x spacing because axes contains zyx
        "units": ("micrometer", "micrometer", "micrometer"),
        "time_increment": 60.0,
        "time_increment_unit": "second",
        "channel_names": ["DAPI", "GFP", "RFP"],
        "channel_colors": ["0000FF", "00FF00", "FF0000"],
        "dtype": "uint16",
        "data_type": "intensity",
    }
    manager = mm.ArrayManager(raw, metadata, chunks=metadata["chunksize"][0])
    if num_levels > 1:
        manager.build_pyramid(num_levels=num_levels, downscale_factor=(1, 2, 2))
    return manager

## Approach 1: write a complete manager to OME-Zarr

In [ ]:
raw_manager = make_small_raw_manager(seed=3, num_levels=3)
summarize_manager(raw_manager, "source manager")

complete_path = clean_path(OUTPUT_DIR / "complete_raw_v05.zarr")
raw_manager.to_zarr(
    complete_path,
    ngff_version="0.5",
    zarr_format=3,
    compressor=None,
    overwrite=True,
)
print("Wrote", complete_path)

## Read it back

`ZarrManager` reads v0.5 and v0.4 stores. The root image is available as both `z.raw` and the old aliases `z.data` / `z.metadata`.

In [ ]:
z = mm.ZarrManager(complete_path, mode="r")
summarize_manager(z, "read back with ZarrManager")
print("z.raw:", z.raw)
print("z.data is z.raw.data:", z.data is z.raw.data)

## Approach 2: create an empty Zarr from metadata

Here the arrays are created on disk first, but not filled with meaningful image data yet.

In [ ]:
empty_path = clean_path(OUTPUT_DIR / "empty_then_filled.zarr")
empty_metadata = dict(raw_manager.metadata)

z_empty = mm.ZarrManager(
    empty_path,
    mode="w",
    metadata=empty_metadata,
    ngff_version="0.5",
    zarr_format=3,
)
print("Created empty Zarr:", empty_path)
summarize_manager(z_empty, "empty zarr manager")

## Fill the root image with a region write

The data shape must match the selected region at the level you are writing. If you pass a single finest-level array, PyMIF generates the lower-resolution levels automatically.

In [ ]:
root_data = raw_manager.data[0].compute()
z_empty.write_image_region(
    root_data,
    t=slice(None), c=slice(None), z=slice(None), y=slice(None), x=slice(None),
    level=0,
    group=None,
    downscale_factor=(1, 2, 2),
)

# Re-read to refresh the manager's in-memory handles.
z_empty.read()
print("Filled root image.")
summarize_manager(z_empty, "filled empty Zarr")

## Legacy v0.4 output when needed

New projects should usually use v0.5/v3. If another tool requires NGFF v0.4, write `ngff_version="0.4", zarr_format=2`.

In [ ]:
legacy_path = clean_path(OUTPUT_DIR / "complete_raw_v04.zarr")
raw_manager.to_zarr(legacy_path, ngff_version="0.4", zarr_format=2)
legacy = mm.ZarrManager(legacy_path, mode="r")
summarize_manager(legacy, "v0.4 read by ZarrManager")